# 2 つの文の含意関係を判定する (NLI)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/15_transformers/nli.ipynb)

NLP クックブック 15-17 ／ Transformers の基本操作

前提文から仮説文が「言える (含意)」「矛盾する」「どちらでもない (中立)」のどれかを判定したい。

In [1]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

name = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"   # 多言語 NLI モデル
tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModelForSequenceClassification.from_pretrained(name).eval()
print(model.config.id2label)

def nli(premise, hypothesis):
    enc = tokenizer(premise, hypothesis, return_tensors="pt", truncation=True)
    with torch.no_grad():
        probs = model(**enc).logits.softmax(-1)[0]
    return {model.config.id2label[i]: round(p.item(), 3) for i, p in enumerate(probs)}

premise = "男性が公園でギターを弾いている。"
for hyp in ["人が楽器を演奏している。", "男性が屋外にいる。", "男性は家で寝ている。", "男性はプロの音楽家だ。", "A man is playing music."]:
    r = nli(premise, hyp)
    print(max(r, key=r.get), r, hyp)

{0: 'entailment', 1: 'neutral', 2: 'contradiction'}
entailment {'entailment': 0.999, 'neutral': 0.001, 'contradiction': 0.0} 人が楽器を演奏している。
entailment {'entailment': 0.999, 'neutral': 0.001, 'contradiction': 0.0} 男性が屋外にいる。
contradiction {'entailment': 0.0, 'neutral': 0.0, 'contradiction': 1.0} 男性は家で寝ている。
neutral {'entailment': 0.002, 'neutral': 0.999, 'contradiction': 0.0} 男性はプロの音楽家だ。
entailment {'entailment': 0.992, 'neutral': 0.008, 'contradiction': 0.0} A man is playing music.


In [2]:
# 応用: 要約や生成文が元の文書と矛盾していないかのチェック (事実整合性)
source = "同社の 2023 年度の売上高は前年比 5% 増の 120 億円だった。営業利益は 8 億円で、前年から減少した。"
for claim in ["売上高は増加した。", "営業利益は増加した。", "同社は海外に進出した。"]:
    r = nli(source, claim)
    print(max(r, key=r.get), claim)

entailment 売上高は増加した。
contradiction 営業利益は増加した。
neutral 同社は海外に進出した。


- NLI は「ゼロショット分類」「事実整合性の評価」「言い換えの検証」などの部品として広く使われる。
- 語彙の重なりが大きいと含意と判定しやすい、といった表面的な手がかりへの依存 (アーティファクト) が知られている。過信は禁物。
- 日本語のデータセットには JNLI (JGLUE)、JSICK、JaNLI などがある。